In [1]:
import sys
!{sys.executable} -m pip install -q 'scanpy[leiden]'


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: /n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/bin/python -m pip install --upgrade pip


In [2]:
import scanpy as sc
import scipy as sp
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pylab as pl
import matplotlib.pyplot as plt
import ot

## 0. Define helper functions

In [ ]:
def sample_cells_by_type(adata, cell_type_key="cell_type", max_per_type=1000, random_state=0):
    """
    Sample up to `max_per_type` cells per cell type from an AnnData object.

    Parameters
    ----------
    adata : AnnData
        Input AnnData object.
    cell_type_key : str
        Column in adata.obs containing cell type labels.
    max_per_type : int
        Maximum number of cells to sample per type.
    random_state : int
        Random seed.

    Returns
    -------
    AnnData
        Subset AnnData with sampled cells.
    """
    rng = np.random.default_rng(random_state)
    sampled = []

    for ct, idx in adata.obs.groupby(cell_type_key).indices.items():
        n_available = len(idx)
        n_sample = min(max_per_type, n_available)
        pick = rng.choice(idx, size=n_sample, replace=False)
        sampled.extend(pick)

    return adata[sampled].copy()

## 1. Generate gene lists, balanced datasets, and cell type mappings between mouse and human

#### Mouse genes and cell types

In [3]:
mouse_dir = 'data/tabula_muris/'

In [4]:
mouse_all_adata = sc.read_h5ad(mouse_dir+'tabula_muris_all.h5ad')

In [ ]:
mouse_genes = mouse_all_adata.var_names
mouse_genes_series = pd.Series(mouse_genes, name='genes')
mouse_genes_series.to_csv(mouse_dir+'tm_genes.csv')

In [ ]:
mouse_cell_types = mouse_all_adata.obs.cell_type.unique()
mouse_cell_types_series = pd.Series(mouse_cell_types, name='cell_types')
mouse_cell_types_series.to_csv(mouse_dir+'tm_cell_types.csv')

In [7]:
mouse_sampled = sample_cells_by_type(mouse_all_adata, cell_type_key="cell_type", max_per_type=1000, random_state=42)
mouse_sampled.write_h5ad(mouse_dir+'sampled_mouse_1000.h5ad')

/tmp/ipykernel_3677670/1096216483.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for ct, idx in adata.obs.groupby(cell_type_key).indices.items():


#### Human genes and cell types

In [9]:
human_dir = 'data/tabula_sapiens/'

In [5]:
human_cell_types = set()
#cell_groups = ['endothelial', 'epithelial', 'germline', 'immune', 'neural', 'stromal']
cell_groups=['immune']
for cell_group in cell_groups:
    cell_group_adata = sc.read_h5ad(human_dir+'tabula_sapiens_'+cell_group+'.h5ad')
    group_cell_types = cell_group_adata.obs.cell_type.unique()
    for cell_type in group_cell_types:
        human_cell_types.add(cell_type)
    group_cell_types_series = pd.Series(group_cell_types, name='cell_types')
    group_cell_types_series.to_csv(human_dir+cell_group+'_ts_cell_types.csv')
    
    group_sampled = sample_cells_by_type(cell_group_adata, cell_type_key="cell_type", max_per_type=1000, random_state=42)
    group_sampled.write_h5ad(human_dir+cell_group+'_sampled_human_1000.h5ad')


#human_cell_types_series = pd.Series(human_cell_types, name='cell_types')
#human_cell_types_series.to_csv(human_dir+'ts_cell_types.csv')

/tmp/ipykernel_3661965/1096216483.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for ct, idx in adata.obs.groupby(cell_type_key).indices.items():


In [ ]:
human_all_adata = sc.read_h5ad(human_dir+'tabula_sapiens_all.h5ad')

In [ ]:
human_genes = human_all_adata.var_names
human_genes_series = pd.Series(human_genes, name='genes')
human_genes_series.to_csv(human_dir+'ts_genes.csv')

In [ ]:
human_cell_types = human_all_adata.obs.cell_type.unique()
human_cell_types_series = pd.Series(human_cell_types, name='cell_types')
human_cell_types_series.to_csv(human_dir+'ts_cell_types.csv')

In [ ]:
human_sampled = sample_cells_by_type(human_all_adata, cell_type_key="cell_type", max_per_type=1000, random_state=42)
human_sampled.write_h5ad(human_dir+'sampled_human_1000.h5ad')

In [26]:
# full human cell type list
cell_groups = ['endothelial', 'epithelial', 'germline', 'immune', 'neural', 'stromal']
human_cell_types_set = set()
for cell_group in cell_groups:
    group_cell_types = pd.read_csv(human_dir+cell_group+'_ts_cell_types.csv')['cell_types'].tolist()
    for cell_type in group_cell_types:
        human_cell_types_set.add(cell_type)

human_cell_types = list(human_cell_types)
human_cell_types_series = pd.Series(human_cell_types, name='cell_types')
human_cell_types_series.to_csv(human_dir+'ts_cell_types.csv')

In [33]:
# make dataframe with 1000 of each human cell type
cell_groups = ['endothelial', 'epithelial', 'germline', 'immune', 'neural', 'stromal']
adata_list = []
for cell_group in cell_groups:
    adata_list += [sc.read_h5ad(human_dir+cell_group+'_sampled_human_1000.h5ad')]
    
human_1000_samples = ad.concat(adata_list)

In [34]:
human_sampled = sample_cells_by_type(human_1000_samples)
human_sampled.write_h5ad(human_dir+'sampled_human_1000.h5ad')

/tmp/ipykernel_3661965/1096216483.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for ct, idx in adata.obs.groupby(cell_type_key).indices.items():


### Making cell type mapping (list of lists -> dataframe)

In [3]:
cell_type_map = [
    ['T cell', ['naive T cell','T cell','CD8-positive, alpha-beta T cell', 'CD4-positive, alpha-beta T cell','regulatory T cell','immature T cell','mature alpha-beta T cell'],['CD4-positive, alpha-beta T cell','naive thymus-derived CD4-positive, alpha-beta T cell','T cell','CD8-positive, alpha-beta T cell','gamma-delta T cell','naive thymus-derived CD8-positive, alpha-beta T cell','regulatory T cell','activated CD8-positive, alpha-beta T cell','activated CD4-positive, alpha-beta T cell']],
    ['B cell', ['precursor B cell','immature B cell','naive B cell','B cell'],['B cell']],
    ['NK cell',['natural killer cell'],['natural killer cell']],
    ['mast cell',['mast cell'],['mast cell']],
    ['erythrocyte', ['erythrocyte'],['erythrocyte']],
    ['HPSC', ['hematopoietic precursor cell','hematopoietic stem cell'],['hematopoietic precursor cell','hematopoietic stem cell']],
    ['dendritic cell', ['dendritic cell','myeloid dendritic cell','plasmacytoid dendritic cell'],['myeloid dendritic cell','plasmacytoid dendritic cell']],
    ['macrophage',['macrophage','alveolar macrophage','lung macrophage','Langerhans cell'],['tissue-resident macrophage','colon macrophage','macrophage']],
    ['plasma cell',['plasma cell'],['plasma cell']],
    ['monocyte',['monocyte','non-classical monocyte','classical monocyte','intermediate monocyte'],['monocyte','non-classical monocyte','classical monocyte','intermediate monocyte']],
    ['neutrophil',['neutrophil'],['neutrophil']],
    ['basophil',['basophil'],['basophil']],
    ['fibroblast',['fibroblast of cardiac tissue','fibroblast','pulmonary interstitial fibroblast','fibroblast of lung','kidney interstitial fibroblast'],['fibroblast','thymic fibroblast type 1','thymic fibroblast type 2','fibroblast of breast','fibroblast of cardiac tissue','alveolar adventitial fibroblast']],
    ['smooth muscle cell',['bronchial smooth muscle cell','smooth muscle cell of the pulmonary artery','smooth muscle cell of trachea'],['bronchial smooth muscle cell','blood vessel smooth muscle cell','smooth muscle cell','vascular associated smooth muscle cell']],
    ['neuron',['cardiac neuron','interneuron','neuron','medium spiny neuron'],['neuron','retinal bipolar neuron']],
    ['endothelial cell',['endothelial cell of coronary artery','fenestrated endothelial cell','kidney capillary endothelial cell','endothelial cell of hepatic sinusoid','vein endothelial cell','aortic endothelial cell'],['endothelial cell of venule','cardiac endothelial cell','colon endothelial cell','capillary endothelial cell','endothelial cell of artery','endothelial cell of vascular tree','vein endothelial cell','endothelial cell of arteriole','retinal blood vessel endothelial cell','endothelial cell','endothelial cell of lymphatic vessel']],
    ['pericyte',['pericyte','brain pericyte'],['pericyte']],
    ['mesenchymal stem cell',['mesenchymal stem cell'],['mesenchymal stem cell','mesenchymal stem cell of adipose tissue']],
    ['Schwann cell',['Schwann cell'],['Schwann cell']],
    ['pancreatic alpha cell',['pancreatic A cell'],['pancreatic A cell']],
    ['pancreatic beta cell',['type B pancreatic cell'],['type B pancreatic cell']],
    ['pancreatic delta cell',['pancreatic D cell'],['pancreatic D cell']],
    ['pancreatic gamma cell',['pancreatic PP cell'],['pancreatic PP cell']],
    ['pancreatic ductal cell',['pancreatic ductal cell'],['pancreatic ductal cell']],
    ['pancreatic acinar cell',['pancreatic acinar cell'],['pancreatic acinar cell']],
    ['thymocyte',['thymocyte','DN3 thymocyte','double negative thymocyte','DN4 thymocyte'],['thymocyte','CD8-positive, alpha-beta thymocyte','CD4-positive, alpha-beta thymocyte']],
    ['bladder urothelial cell',['bladder urothelial cell'],['bladder urothelial cell']],
    ['goblet cell',['large intestine goblet cell','mucus secreting cell'],['large intestine goblet cell','respiratory tract goblet cell','small intestine goblet cell','tracheal goblet cell']],
    ['microglial cell',['microglial cell'],['microglial cell']],
    ['myocyte',['regular atrial cardiac myocyte','regular ventricular cardiac myocyte'],['regular atrial cardiac myocyte']],
    ['pulmonary alveolar cell',['pulmonary alveolar type 1 cell','pulmonary alveolar type 2 cell'],['pulmonary alveolar type 1 cell','pulmonary alveolar type 2 cell']]
]

In [4]:
cell_type_table = pd.DataFrame(cell_type_map, columns=['shared_cell_type','mouse_cell_types','human_cell_types'])

In [5]:
cell_type_table.to_csv('data/tabula_cell_type_table.csv')

In [ ]:
# Removed due to no representation in human cell types
#['keratinocyte',['keratinocyte'],[]]
#['epidermal cell',['epidermal cell'],[]],
#['neuroendocrine cell',['neuroendocrine cell','pulmonary neuroendocrine cell'],[]],
#['chondrocyte',['chondrocyte'],[]],
#['oligodendrocyte',['oligodendrocyte'],[]],
#['astrocyte',['astrocyte'],[]],

# Helper functions (run these first, in order)